# COMPX525 A2 Synthetic Audio Generation

Generate planned synthetic training audio with AudioLDM2.


In [ ]:
from __future__ import annotations

import csv
import json
import subprocess
import sys
import time
from pathlib import Path

print("Installing generation dependencies")
if True:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "torch==2.5.1+cu121",
        "torchaudio==2.5.1+cu121",
        "--index-url",
        "https://download.pytorch.org/whl/cu121",
    ])
    # Kaggle's preinstalled torchvision is pinned to the default torch build.
    # AudioLDM2 does not need torchvision, but transformers probes it during
    # CLAP imports; a mismatched torchvision can fail import with
    # "operator torchvision::nms does not exist".
    subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision"])
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--upgrade",
    "diffusers>=0.35.1",
    "transformers",
    "accelerate",
    "scipy",
    "soundfile",
    "sentencepiece",
])

import numpy as np
import scipy.io.wavfile
import torch
from diffusers import AudioLDM2Pipeline

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
PLAN_PATH = Path("synthetic_plan.csv")
OUTPUT_AUDIO_ROOT = WORK_DIR / "synthetic_audio"
OUTPUT_AUDIO_ROOT.mkdir(parents=True, exist_ok=True)

START = 0
LIMIT = 6
MODEL_ID = 'cvssp/audioldm2'
NUM_INFERENCE_STEPS = 25
GUIDANCE_SCALE = 3.5
AUDIO_LENGTH_IN_S = 5.0

with PLAN_PATH.open(newline="", encoding="utf-8") as f:
    all_rows = list(csv.DictReader(f))

rows = all_rows[START:START + LIMIT]
print(json.dumps({
    "plan_rows_total": len(all_rows),
    "start": START,
    "limit": LIMIT,
    "selected_rows": len(rows),
    "model_id": MODEL_ID,
    "num_inference_steps": NUM_INFERENCE_STEPS,
    "guidance_scale": GUIDANCE_SCALE,
    "audio_length_in_s": AUDIO_LENGTH_IN_S,
}, indent=2))

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print(f"torch={torch.__version__} cuda_available={torch.cuda.is_available()} device={device}")

pipe = AudioLDM2Pipeline.from_pretrained(MODEL_ID, torch_dtype=dtype)
if device == "cuda":
    try:
        pipe.enable_model_cpu_offload()
        generator_device = "cuda"
        print("Enabled model CPU offload")
    except Exception as exc:
        print(f"CPU offload unavailable: {exc}; using pipe.to(cuda)")
        pipe = pipe.to("cuda")
        generator_device = "cuda"
else:
    pipe = pipe.to("cpu")
    generator_device = "cpu"

generated_rows = []
started = time.time()

for n, row in enumerate(rows, start=1):
    item_started = time.time()
    seed = int(row["seed"])
    generator = torch.Generator(device=generator_device).manual_seed(seed)
    prompt = row["prompt"]
    negative_prompt = row.get("negative_prompt") or None
    rel_path = Path(row["relative_audio_path"])
    out_path = WORK_DIR / rel_path
    out_path.parent.mkdir(parents=True, exist_ok=True)

    print(f"[{n}/{len(rows)}] {row['synthetic_id']} label={row['primary_label']} seed={seed}")
    try:
        result = pipe(
            prompt,
            negative_prompt=negative_prompt,
            num_inference_steps=NUM_INFERENCE_STEPS,
            guidance_scale=GUIDANCE_SCALE,
            audio_length_in_s=AUDIO_LENGTH_IN_S,
            num_waveforms_per_prompt=1,
            generator=generator,
        )
        audio = np.asarray(result.audios[0], dtype=np.float32)
        audio = np.nan_to_num(audio, nan=0.0, posinf=0.0, neginf=0.0)
        audio = np.clip(audio, -1.0, 1.0)
        scipy.io.wavfile.write(out_path, rate=16000, data=audio)
        row["generated_audio_path"] = str(out_path)
        row["filter_status"] = "generated_pending_birdnet_filter"
        row["filter_reason"] = ""
        row["generation_elapsed_seconds"] = round(time.time() - item_started, 3)
    except Exception as exc:
        row["generated_audio_path"] = ""
        row["filter_status"] = "generation_failed"
        row["filter_reason"] = repr(exc)
        row["generation_elapsed_seconds"] = round(time.time() - item_started, 3)
        print(f"FAILED {row['synthetic_id']}: {exc!r}")
    generated_rows.append(row)

manifest_path = WORK_DIR / "synthetic_generation_manifest.csv"
fieldnames = list(generated_rows[0].keys()) if generated_rows else list(all_rows[0].keys())
with manifest_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(generated_rows)

summary = {
    "plan_rows_total": len(all_rows),
    "start": START,
    "limit": LIMIT,
    "selected_rows": len(rows),
    "generated_count": sum(r.get("filter_status") == "generated_pending_birdnet_filter" for r in generated_rows),
    "failed_count": sum(r.get("filter_status") == "generation_failed" for r in generated_rows),
    "elapsed_seconds": round(time.time() - started, 3),
    "model_id": MODEL_ID,
    "num_inference_steps": NUM_INFERENCE_STEPS,
    "guidance_scale": GUIDANCE_SCALE,
    "audio_length_in_s": AUDIO_LENGTH_IN_S,
    "manifest_path": str(manifest_path),
}
(WORK_DIR / "synthetic_generation_summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True), encoding="utf-8")
print(json.dumps(summary, indent=2))
print("Synthetic generation complete")
